# Unified Training Pipeline

This notebook integrates data preprocessing and training of a ResNet for both binary and multiclass motor fault classification.

The pipeline:
1. Loads raw oscilloscope data from multiple files.
2. Processes data in memory (FFT, segmentation, normalization).
3. Computes anomaly peak frequencies from the engine configuration.
4. Generates synthetic peaks and inserts them into real data at all computed frequencies.
5. Constructs datasets for binary and multiclass classification.
6. Trains a ResNet model using the combined real and synthetic data.

In [1]:
# Logging
import comet_ml
import yaml
import datetime

with open("../cfg.yaml", "r") as f:
    config = yaml.safe_load(f)

api_key = config['API_KEY']
project_name = config['PROJECT_NAME']
workspace = config['WORKSPACE']

experiment = comet_ml.start(api_key=api_key, 
                        project_name=project_name, 
                        workspace=workspace)

current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(current_time)
experiment.set_name(f"Run: {current_time}")
experiment.log_code()

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/xndrleib/motor-fault-detection/e0ddb04e65e747fab53637085c6e6719

COMET WARNING: unable to find caller source code in a jupyter notebook; ignoring


2025-02-28 14:25:08


In [2]:
import numpy as np

import torch

import random
from src.data_preprocessing import (
    generate_synthetic_peak, process_file, insert_synthetic_peaks, create_dataloaders, create_dataset
)

from src.electrical_signature_frequencies import ANOMALY_FREQS
from src.models import ResNet, ResidualBlock
from src.train import train_resnet_model, inference_resnet_model, calculate_metrics

from sklearn.metrics import classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Set device 
device: str = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
experiment.log_parameter("device", device)
device = torch.device(device)
print("Using device:", device)

Using device: mps


## 1. Load Engine Configuration and Set Parameters

In [3]:
with open('../engine_configs/LIMAN.yml', 'r') as f:
    engine_config = yaml.safe_load(f)

# Processing Parameters
f_sampling = 10000
shift = 20
segment_length = 10000
cutoff_freq = 250
peak_type = 'gaussian'
peak_segment = 2  # +- 2 Hz
amplitude_range = (2.0, 20.0)


# Training Parameters
num_epochs_binary = 10
num_epochs_multi = 10

# Data Parameters
batch_size = 16

# Define a mapping from raw file paths to labels.
# For multiclass, each file can have its specific fault label.
file_label_map = {
    '../data/raw/0702 без перекосов фаза A.txt': 'normal',
    '../data/raw/0702  межвитковые фазы A ток фазы А.txt': 'inter-turn short circuits',
    '../data/raw/20_01_1460_об_мин_с_дисбалансом_ротора.txt': 'rotor bar defect'
}

# For synthetic injection, provide a list of fault types to choose from
fault_types_available = [
    'rotor bar defect',
    # 'air-gap eccentricity',
    'inter-turn short circuits'
#     'bearing defect (rolling element)',
#     'bearing defect (outer race)',
#     'bearing defect (inner race)',
#     'other mechanical defects'
]

experiment.log_parameters({
    "f_sampling": f_sampling,
    "shift": shift,
    "segment_length": segment_length,
    "peak_segment": peak_segment,
    "cutoff_freq": cutoff_freq,
    "file_label_map": file_label_map,
    "fault_types_available": fault_types_available,
    "peak_type": peak_type,
    "num_epochs_binary": num_epochs_binary,
    "num_epochs_multi": num_epochs_multi,
    "batch_size": batch_size
})

fault_freqs = {fault_type: ANOMALY_FREQS.get(fault_type, lambda ec: [])(engine_config) for fault_type in fault_types_available}
experiment.log_parameters(fault_freqs)

### Balancing Classes

Assume a file labeled "normal" produces $N$ segments. If you inject synthetic faults into a fraction $p$ of them, then:
- The number of unmodified (normal) segments is $(1-p)N$.
- The number of synthetic fault segments is $pN$.

If you have $k$ distinct fault types and you select each fault type uniformly at random during injection, then each fault type will get about $\frac{pN}{k}$ segments on average.

To have balanced classes across the $k+1$ classes (one normal and $k$ faults), you’d want:
$$
(1-p)N \approx \frac{pN}{k}\,,
$$
which simplifies to:
$$
1-p = \frac{p}{k} \quad \Longrightarrow \quad p \left( 1+\frac{1}{k} \right) = 1 \quad \Longrightarrow \quad p = \frac{k}{k+1}\,.
$$

For example:
- In binary classification (only one fault type, $k=1$), $p = \frac{1}{2} = 0.5$.
- In multiclass classification with, say, 3 fault types ($k=3$), $p = \frac{3}{4} = 0.75$.  
  This means that 75% of the segments from a normal file would be synthetically modified, and—if the injection is uniform—the normal class (25% of the segments) and each fault class (approximately 25% each) would be balanced.


In [4]:
# Saving spectrums before and after peak injection

sample_file = None
for fp, lbl in file_label_map.items():
    if lbl == 'normal':
        sample_file = fp
        break

if sample_file is not None:
    try:
        segs, labels, freqs = process_file(sample_file, engine_config, 'normal',
                                            f_sampling=f_sampling, cutoff_freq=cutoff_freq,
                                            segment_length=segment_length, peak_segment=peak_segment, step=shift,
                                            apply_window=False, db=True,
                                            synthetic_fault_fraction=0.0, fault_types=None, amplitude_range=amplitude_range)
    except NameError:
        pass

    if segs is not None and len(segs) > 0:
        sample_segment = segs[0]
        for fault_type in fault_types_available:
            injected_segment = insert_synthetic_peaks(
                segment=sample_segment,
                fft_freqs=freqs,
                fault_type=fault_type,
                engine_config=engine_config,
                peak_segment=peak_segment,
                peak_generator=generate_synthetic_peak,
                sigma=2.0,
                method='gaussian'
            )
            difference = injected_segment - sample_segment
            fig, axs = plt.subplots(1, 3, figsize=(18, 5))
            axs[0].plot(freqs, sample_segment, color='blue')
            axs[0].set_title("Before Processing")
            axs[0].set_xlabel("Frequency")
            axs[0].set_ylabel("Magnitude (dB)")
            axs[0].grid(True)
            
            axs[1].plot(freqs, injected_segment, color='orange')
            axs[1].set_title(f"After Processing ({fault_type})")
            axs[1].set_xlabel("Frequency")
            axs[1].grid(True)
            
            axs[2].plot(freqs, difference, color='green')
            axs[2].set_title("Difference (After - Before)")
            axs[2].set_xlabel("Frequency")
            axs[2].grid(True)
            
            fig.suptitle(f"Spectrum Comparison for {fault_type}")
            experiment.log_figure(figure=fig, figure_name=f"spectrum_comparison_{fault_type}.png")
            plt.close(fig)

## 2. Train the ResNet Binary Classifier

In [5]:
dataset_binary = create_dataset(file_label_map, engine_config, mode="binary",
                                f_sampling=f_sampling, cutoff_freq=cutoff_freq,
                                segment_length=segment_length, step=shift,
                                apply_window=False, db=True,
                                fault_types_available=fault_types_available,
                                include_real_anomalies_in_training=False, amplitude_range=amplitude_range)

print("Binary dataset shapes:", dataset_binary["X_train"].shape, dataset_binary["y_train"].shape, dataset_binary["X_test"].shape, dataset_binary["y_test"].shape)

train_loader_binary, val_loader_binary, test_loader_binary = create_dataloaders(dataset_binary, batch_size=batch_size)

Binary dataset shapes: (2520, 250) (2520,) (3303, 250) (3303,)


In [6]:
# Initialize ResNet
model_bin = ResNet(ResidualBlock, [2, 2, 2, 2], num_classes=2).to(device)
print("Training ResNet for binary classification...")
model_bin = train_resnet_model(model_bin, train_loader_binary, val_loader=val_loader_binary, device=device, num_epochs=num_epochs_binary)

Training ResNet for binary classification...


Epoch 1/10:   0%|          | 0/158 [00:00<?, ?batch/s]

Epoch [1/10], Loss: 0.1682
Validation Loss: 0.2626, Accuracy: 90.04%
Best model updated.


Epoch 2/10:   0%|          | 0/158 [00:00<?, ?batch/s]

Epoch [2/10], Loss: 0.0483
Validation Loss: 0.8942, Accuracy: 69.71%


...

Epoch [8/10], Loss: 0.0068
Validation Loss: 0.9149, Accuracy: 79.21%


Epoch 9/10:   0%|          | 0/158 [00:00<?, ?batch/s]

Epoch [9/10], Loss: 0.0012
Validation Loss: 1.0289, Accuracy: 78.82%


Epoch 10/10:   0%|          | 0/158 [00:00<?, ?batch/s]

Epoch [10/10], Loss: 0.0006
Validation Loss: 0.9520, Accuracy: 79.50%


In [7]:
true_labels_bin, predictions_bin = inference_resnet_model(model_bin, test_loader_binary, device=device)

Inference:   0%|          | 0/207 [00:00<?, ?batch/s]

In [8]:
# Calculate metrics 
cm_bin, acc_bin, prec_bin, rec_bin, f1_bin = calculate_metrics(true_labels_bin, predictions_bin)

In [9]:
binary_report = classification_report(true_labels_bin, predictions_bin, target_names=['normal', 'anomalous'], output_dict=True)
experiment.log_metrics(binary_report)

In [10]:
fig_bin, ax_bin = plt.subplots(figsize=(5,4))
sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Blues',
            xticklabels=['normal','anomalous'], yticklabels=['normal','anomalous'], ax=ax_bin)
ax_bin.set_xlabel('Predicted Label')
ax_bin.set_ylabel('True Label')
ax_bin.set_title('Binary Classification - Confusion Matrix')
experiment.log_figure(figure=fig_bin, figure_name="confusion_matrix_binary.png")
plt.close(fig_bin)

## 3. Train the ResNet Multiclass Classifier

In [11]:
dataset_multi = create_dataset(file_label_map, engine_config, mode="multiclass",
                               f_sampling=f_sampling, cutoff_freq=cutoff_freq,
                               segment_length=segment_length, peak_segment=peak_segment, step=shift,
                               apply_window=False, db=True,
                               fault_types_available=fault_types_available,
                               include_real_anomalies_in_training=False, amplitude_range=amplitude_range)

print("Multiclass dataset shapes:", dataset_multi["X_train"].shape, dataset_multi["y_train"].shape, dataset_multi["X_test"].shape, dataset_multi["y_test"].shape)
train_loader_multi, val_loader_multi, test_loader_multi = create_dataloaders(dataset_multi, batch_size=batch_size)

Multiclass dataset shapes: (2520, 250) (2520,) (3303, 250) (3303,)


In [12]:
# Train ResNet for Multiclass Classification
num_classes_multi = len(dataset_multi['label_encoder'].classes_)
model_multi = ResNet(ResidualBlock, [2, 2, 2, 2], num_classes=num_classes_multi).to(device)
print("Training ResNet for multiclass classification...")
model_multi = train_resnet_model(model_multi, train_loader_multi, val_loader=val_loader_multi, device=device, num_epochs=num_epochs_multi)

Training ResNet for multiclass classification...


Epoch 1/10:   0%|          | 0/158 [00:00<?, ?batch/s]

Epoch [1/10], Loss: 0.2334
Validation Loss: 1.3320, Accuracy: 69.17%
Best model updated.


Epoch 2/10:   0%|          | 0/158 [00:00<?, ?batch/s]

Epoch [2/10], Loss: 0.0617
Validation Loss: 0.9768, Accuracy: 74.17%
Best model updated.


...

Epoch [8/10], Loss: 0.0156
Validation Loss: 2.6872, Accuracy: 60.39%


Epoch 9/10:   0%|          | 0/158 [00:00<?, ?batch/s]

Epoch [9/10], Loss: 0.0029
Validation Loss: 2.9528, Accuracy: 64.42%


Epoch 10/10:   0%|          | 0/158 [00:00<?, ?batch/s]

Epoch [10/10], Loss: 0.0026
Validation Loss: 1.8652, Accuracy: 70.39%


In [13]:
true_labels_multi, predictions_multi = inference_resnet_model(model_multi, test_loader_multi, device=device)

Inference:   0%|          | 0/207 [00:00<?, ?batch/s]

In [14]:
# Calculate metrics 
cm_multi, acc_multi, prec_multi, rec_multi, f1_multi = calculate_metrics(true_labels_multi, predictions_multi)

In [15]:
multiclass_report = classification_report(true_labels_multi, predictions_multi, target_names=dataset_multi['label_encoder'].classes_, output_dict=True)
experiment.log_metrics(multiclass_report)

In [16]:
fig_multi, ax_multi = plt.subplots(figsize=(5,4))
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues',
            xticklabels=dataset_multi['label_encoder'].classes_,
            yticklabels=dataset_multi['label_encoder'].classes_, ax=ax_multi)
ax_multi.set_xlabel('Predicted Label')
ax_multi.set_ylabel('True Label')
ax_multi.set_title('Multiclass Classification - Confusion Matrix')
experiment.log_figure(figure=fig_multi, figure_name="confusion_matrix_multiclass.png")
plt.close(fig_multi)

In [17]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : Run: 2025-02-28 14:25:08
COMET INFO:     url                   : https://www.comet.com/xndrleib/motor-fault-detection/e0ddb04e65e747fab53637085c6e6719
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     accuracy [2]                        : (0.7768695125643355, 0.8652739933393885)
COMET INFO:     anomalous/f1-score                  : 0.8714244438023693
COMET INFO:     anomalous/precision                 : 0.9993373094764745
COMET INFO:     anomalous/recall                    : 0.7725409836065574
COMET INFO:     anomalous/support                   : 1952.0
COMET INFO:     inter-turn short circuits/f1-score  : 0.03273322422258593
COMET INFO:     